In [43]:
import pandas as pd

matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print(matches.shape, deliveries.shape)

(756, 18) (179078, 21)


In [44]:
print(matches.columns.tolist())


['id', 'Season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']


In [45]:
print(deliveries.columns.tolist())

['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']


In [46]:
print(deliveries.head(3))

   match_id  inning         batting_team                 bowling_team  over  \
0         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
1         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
2         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   

   ball    batsman non_striker    bowler  is_super_over  ...  bye_runs  \
0     1  DA Warner    S Dhawan  TS Mills              0  ...         0   
1     2  DA Warner    S Dhawan  TS Mills              0  ...         0   
2     3  DA Warner    S Dhawan  TS Mills              0  ...         0   

   legbye_runs  noball_runs  penalty_runs  batsman_runs  extra_runs  \
0            0            0             0             0           0   
1            0            0             0             0           0   
2            0            0             0             4           0   

   total_runs  player_dismissed dismissal_kind fielder  
0           0               NaN            N

In [47]:
print(deliveries.isnull().sum())

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batsman                  0
non_striker              0
bowler                   0
is_super_over            0
wide_runs                0
bye_runs                 0
legbye_runs              0
noball_runs              0
penalty_runs             0
batsman_runs             0
extra_runs               0
total_runs               0
player_dismissed    170244
dismissal_kind      170244
fielder             172630
dtype: int64


In [48]:
print(deliveries['batting_team'].unique())

['Sunrisers Hyderabad' 'Royal Challengers Bangalore' 'Mumbai Indians'
 'Rising Pune Supergiant' 'Gujarat Lions' 'Kolkata Knight Riders'
 'Kings XI Punjab' 'Delhi Daredevils' 'Chennai Super Kings'
 'Rajasthan Royals' 'Deccan Chargers' 'Kochi Tuskers Kerala'
 'Pune Warriors' 'Rising Pune Supergiants' 'Delhi Capitals']


In [49]:
print(deliveries['over'].unique())

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]


In [50]:
print(deliveries[['batting_team','bowling_team','over','ball','batsman_runs','extra_runs','total_runs']].head(10))

          batting_team                 bowling_team  over  ball  batsman_runs  \
0  Sunrisers Hyderabad  Royal Challengers Bangalore     1     1             0   
1  Sunrisers Hyderabad  Royal Challengers Bangalore     1     2             0   
2  Sunrisers Hyderabad  Royal Challengers Bangalore     1     3             4   
3  Sunrisers Hyderabad  Royal Challengers Bangalore     1     4             0   
4  Sunrisers Hyderabad  Royal Challengers Bangalore     1     5             0   
5  Sunrisers Hyderabad  Royal Challengers Bangalore     1     6             0   
6  Sunrisers Hyderabad  Royal Challengers Bangalore     1     7             0   
7  Sunrisers Hyderabad  Royal Challengers Bangalore     2     1             1   
8  Sunrisers Hyderabad  Royal Challengers Bangalore     2     2             4   
9  Sunrisers Hyderabad  Royal Challengers Bangalore     2     3             0   

   extra_runs  total_runs  
0           0           0  
1           0           0  
2           0           

In [51]:
df = deliveries[(deliveries['inning'] == 1) & (deliveries['is_super_over'] == 0)]


In [52]:
df = df.groupby(['match_id', 'inning', 'batting_team', 'bowling_team', 'over']).agg(
    runs_in_over = ('total_runs', 'sum'),
    wickets_in_over = ('player_dismissed', 'count')
).reset_index()

In [53]:
print(df.head(10))
print(df.shape)

   match_id  inning         batting_team                 bowling_team  over  \
0         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     1   
1         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     2   
2         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     3   
3         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     4   
4         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     5   
5         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     6   
6         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     7   
7         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     8   
8         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     9   
9         1       1  Sunrisers Hyderabad  Royal Challengers Bangalore    10   

   runs_in_over  wickets_in_over  
0             7                0  
1            16                1  
2             6          

In [55]:
df['cumulative_runs'] = df.groupby('match_id')['runs_in_over'].cumsum()
df['cumulative_wickets'] = df.groupby('match_id')['wickets_in_over'].cumsum()
final_scores = df.groupby('match_id')['cumulative_runs'].max().reset_index()
final_scores.columns = ['match_id', 'final_score']


In [56]:
df = df.merge(final_scores, on='match_id')

print(df[['batting_team', 'bowling_team', 'over', 'cumulative_runs', 'cumulative_wickets', 'final_score']].head(10))

          batting_team                 bowling_team  over  cumulative_runs  \
0  Sunrisers Hyderabad  Royal Challengers Bangalore     1                7   
1  Sunrisers Hyderabad  Royal Challengers Bangalore     2               23   
2  Sunrisers Hyderabad  Royal Challengers Bangalore     3               29   
3  Sunrisers Hyderabad  Royal Challengers Bangalore     4               33   
4  Sunrisers Hyderabad  Royal Challengers Bangalore     5               42   
5  Sunrisers Hyderabad  Royal Challengers Bangalore     6               59   
6  Sunrisers Hyderabad  Royal Challengers Bangalore     7               64   
7  Sunrisers Hyderabad  Royal Challengers Bangalore     8               75   
8  Sunrisers Hyderabad  Royal Challengers Bangalore     9               84   
9  Sunrisers Hyderabad  Royal Challengers Bangalore    10               88   

   cumulative_wickets  final_score  
0                   0          207  
1                   1          207  
2                   1         

In [57]:

old_teams = [
    'Deccan Chargers', 'Kochi Tuskers Kerala', 
    'Pune Warriors', 'Rising Pune Supergiant',
    'Rising Pune Supergiants', 'Gujarat Lions',
    'Delhi Daredevils'
]

df = df[~df['batting_team'].isin(old_teams)]
df = df[~df['bowling_team'].isin(old_teams)]

print("Remaining teams:", df['batting_team'].unique())
print("Remaining rows:", df.shape)

Remaining teams: ['Sunrisers Hyderabad' 'Kolkata Knight Riders'
 'Royal Challengers Bangalore' 'Kings XI Punjab' 'Mumbai Indians'
 'Chennai Super Kings' 'Rajasthan Royals' 'Delhi Capitals']
Remaining rows: (8634, 10)


In [58]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error

In [59]:
le_batting = LabelEncoder()
le_bowling = LabelEncoder()
df['batting_team_encoded'] = le_batting.fit_transform(df['batting_team'])
df['bowling_team_encoded'] = le_bowling.fit_transform(df['bowling_team'])


In [61]:
features = ['batting_team_encoded', 'bowling_team_encoded', 
            'over', 'cumulative_runs', 'cumulative_wickets']
X = df[features]           
y = df['final_score']   


In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [63]:
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


Training rows: 6907
Testing rows: 1727


In [64]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [65]:
y_pred = model.predict(X_test)
print("\nR² Score:", round(r2_score(y_test, y_pred), 4))
print("Mean Absolute Error:", round(mean_absolute_error(y_test, y_pred), 2), "runs")


R² Score: 0.5216
Mean Absolute Error: 14.65 runs


In [66]:
df_model = df[df['over'] >= 6].copy()


In [67]:
df_model['current_run_rate'] = df_model['cumulative_runs'] / df_model['over']
df_model['remaining_overs'] = 20 - df_model['over']
df_model['runs_per_wicket'] = df_model['cumulative_runs'] / (df_model['cumulative_wickets'] + 1)

print(df_model[['over', 'cumulative_runs', 'current_run_rate', 
                 'remaining_overs', 'runs_per_wicket', 'final_score']].head(10))
print("\nRows remaining:", df_model.shape[0])

    over  cumulative_runs  current_run_rate  remaining_overs  runs_per_wicket  \
5      6               59          9.833333               14        29.500000   
6      7               64          9.142857               13        32.000000   
7      8               75          9.375000               12        37.500000   
8      9               84          9.333333               11        42.000000   
9     10               88          8.800000               10        44.000000   
10    11               98          8.909091                9        32.666667   
11    12              106          8.833333                8        35.333333   
12    13              124          9.538462                7        41.333333   
13    14              132          9.428571                6        44.000000   
14    15              151         10.066667                5        50.333333   

    final_score  
5           207  
6           207  
7           207  
8           207  
9           207  


In [68]:
features_v2 = [
    'batting_team_encoded', 
    'bowling_team_encoded',
    'over', 
    'cumulative_runs', 
    'cumulative_wickets',
    'current_run_rate',
    'remaining_overs',
    'runs_per_wicket'
]

In [69]:
X = df_model[features_v2]
y = df_model['final_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [70]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("R² Score:", round(r2_score(y_test, y_pred), 4))
print("Mean Absolute Error:", round(mean_absolute_error(y_test, y_pred), 2), "runs")

# See actual vs predicted side by side
import pandas as pd
results = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred.round()})
print("\nSample predictions:")
print(results.head(10))

R² Score: 0.6987
Mean Absolute Error: 11.81 runs

Sample predictions:
   Actual  Predicted
0     188      173.0
1     139      136.0
2     175      170.0
3     192      208.0
4     140      156.0
5     194      185.0
6     182      173.0
7      70       71.0
8     136      144.0
9     177      172.0


In [71]:
# Get unique match ids
match_ids = df_model['match_id'].unique()

# Split match IDs into 80/20
from sklearn.model_selection import train_test_split
import numpy as np

train_matches, test_matches = train_test_split(
    match_ids, test_size=0.2, random_state=42
)

# Now split data based on match ids
train_data = df_model[df_model['match_id'].isin(train_matches)]
test_data = df_model[df_model['match_id'].isin(test_matches)]

features_v2 = [
    'batting_team_encoded', 
    'bowling_team_encoded',
    'over', 
    'cumulative_runs', 
    'cumulative_wickets',
    'current_run_rate',
    'remaining_overs',
    'runs_per_wicket'
]

X_train = train_data[features_v2]
y_train = train_data['final_score']

X_test = test_data[features_v2]
y_test = test_data['final_score']

# Retrain
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("R² Score:", round(r2_score(y_test, y_pred), 4))
print("Mean Absolute Error:", round(mean_absolute_error(y_test, y_pred), 2), "runs")

# Sample predictions
import pandas as pd
results = pd.DataFrame({
    'Actual': y_test.values, 
    'Predicted': y_pred.round().astype(int)
})
print("\nSample predictions:")
print(results.head(15))

R² Score: 0.6224
Mean Absolute Error: 14.87 runs

Sample predictions:
    Actual  Predicted
0      207        198
1      207        195
2      207        203
3      207        195
4      207        201
5      207        203
6      207        204
7      207        206
8      207        205
9      207        201
10     207        206
11     207        196
12     207        210
13     207        205
14     207        209


In [72]:
import pickle

# Save the model and encoders so Streamlit can use them
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('batting_encoder.pkl', 'wb') as f:
    pickle.dump(le_batting, f)

with open('bowling_encoder.pkl', 'wb') as f:
    pickle.dump(le_bowling, f)

print("Model and encoders saved successfully!")
print("Teams the model knows:", list(le_batting.classes_))

Model and encoders saved successfully!
Teams the model knows: ['Chennai Super Kings', 'Delhi Capitals', 'Kings XI Punjab', 'Kolkata Knight Riders', 'Mumbai Indians', 'Rajasthan Royals', 'Royal Challengers Bangalore', 'Sunrisers Hyderabad']
